# Python Web Scraping and Data Analysis Portfolio

**Author:** Yayson Valencia  
**Program:** MS in Business Analytics, University of Florida

This case study demonstrates two connected capabilities: preparing imperfect business data for analysis and collecting dynamically loaded public web content with Selenium. The notebook uses included CSV outputs so the analytical sections remain reproducible without live web access.

## Academic context

This portfolio originated in **QMB 6938 Bootcamp Portfolio**, a course in the **University of Florida Warrington College of Business MS in Business Analytics program**. I refined the course exercises into this public-facing case study to demonstrate Python, pandas, Selenium, data cleaning, visualization, and responsible web-data collection.

## Objectives

1. Clean and validate a 105-record book catalog.
2. Summarize category, pricing, rating, and inventory patterns.
3. Document a responsible Selenium workflow for dynamically loaded content.
4. Analyze the resulting 40-record quote dataset.
5. State methodological limitations rather than overgeneralizing from small practice datasets.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = Path(".")
BOOKS_PATH = DATA_DIR / "week3_books.csv"
QUOTES_PATH = DATA_DIR / "quotes.csv"

pd.set_option("display.max_columns", 20)

## Part 1: Book catalog data cleaning

The source intentionally contains inconsistent capitalization, missing values, an unused field, and columns requiring explicit type handling.

In [ ]:
books_raw = pd.read_csv(BOOKS_PATH)
print(f"Rows: {len(books_raw):,}")
display(books_raw.head())
display(books_raw.isna().sum().rename("missing_values"))

In [ ]:
books = (
    books_raw
    .drop(columns=["ExtraColumn"])
    .rename(columns={
        "Title": "title",
        "Category": "category",
        "Price": "price",
        "Rating": "rating",
        "InStock": "in_stock",
        "Discount": "discount",
    })
)

for column in ["price", "rating", "discount"]:
    books[column] = pd.to_numeric(books[column], errors="coerce")

books["title"] = books["title"].fillna("Unknown")
books["category"] = books["category"].fillna("Unknown").str.strip().str.title()
books["price"] = books["price"].fillna(books["price"].mean())
books["rating"] = books["rating"].fillna(books["rating"].mean())
books["discount"] = books["discount"].fillna(0)
books["in_stock"] = (
    books["in_stock"].astype("string").str.lower().map({"true": True, "false": False}).fillna(False)
)

assert len(books) == len(books_raw)
assert books.isna().sum().sum() == 0
display(books.head())

### Cleaning rationale

- Mean imputation retains all records for this practice analysis, although it reduces variance.
- Missing categorical values receive an explicit `Unknown` label instead of being silently dropped.
- Missing inventory values are treated as unavailable, a conservative assumption that avoids overstating stock.
- The unused field is removed because it contains no analytical information.

In [ ]:
summary = pd.Series({
    "records": len(books),
    "categories": books["category"].nunique(),
    "average_price": books["price"].mean(),
    "average_rating": books["rating"].mean(),
    "products_in_stock": int(books["in_stock"].sum()),
})
display(summary.to_frame("value").round(2))

category_summary = (
    books.groupby("category", as_index=False)
    .agg(
        products=("title", "count"),
        average_price=("price", "mean"),
        average_rating=("rating", "mean"),
        products_in_stock=("in_stock", "sum"),
    )
    .sort_values("products", ascending=False)
)
display(category_summary.round(2))

In [ ]:
ax = category_summary.plot.bar(
    x="category", y="products", legend=False, color="#2563eb", figsize=(9, 4)
)
ax.set_title("Book records by category")
ax.set_xlabel("")
ax.set_ylabel("Products")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

### Book-data findings

- The cleaned dataset contains 105 records across six labels, including `Unknown`.
- The average price is approximately $18.53 and the average rating is 3.25 out of 5.
- Only 38 products are explicitly identified as in stock after applying the conservative missing-value rule.
- Sci-Fi is the largest represented category with 26 products. This describes the supplied dataset, not the broader book market.

## Part 2: Dynamic web scraping with Selenium

The standalone `scrape_quotes.py` script opens the JavaScript-rendered scrolling page at `quotes.toscrape.com`, performs three controlled scroll cycles, extracts quote text and author names, removes duplicates, and exports the results to `data/quotes.csv`.

The site is designed for scraping practice. For any production target, review its terms, robots directives, rate expectations, privacy implications, and applicable law before collecting content.

In [ ]:
quotes = pd.read_csv(QUOTES_PATH)
quotes["quote_length"] = quotes["quote"].str.len()

assert quotes[["quote", "author"]].isna().sum().sum() == 0
assert not quotes.duplicated(subset=["quote", "author"]).any()

print(f"Quote records: {len(quotes):,}")
print(f"Unique authors: {quotes['author'].nunique():,}")
print(f"Average quote length: {quotes['quote_length'].mean():.1f} characters")
display(quotes.head())

In [ ]:
author_counts = quotes["author"].value_counts().head(10)
display(author_counts.rename("quotes").to_frame())

ax = author_counts.sort_values().plot.barh(color="#7c3aed", figsize=(8, 5))
ax.set_title("Most represented authors in the collected sample")
ax.set_xlabel("Quotes")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

### Scraping-output findings

- The supplied collection contains 40 complete quotes from 24 unique authors.
- Average quote length is approximately 136 characters.
- Albert Einstein is the most frequently represented author with seven quotes, followed by J.K. Rowling with five.
- These counts reflect the content loaded during the configured scroll cycles. They do not measure author popularity or literary importance.

## Limitations and next steps

- Both datasets are instructional and should not support market-level conclusions.
- Mean imputation can compress the natural spread of price and rating values.
- Treating missing inventory as unavailable may understate actual availability.
- Selenium selectors can break when a target site's HTML changes.
- A production scraper should add explicit retry logic, structured logging, rate controls, and automated tests.
- A larger analytical project could retain quote tags, compare themes, or apply transparent text-analysis methods after verifying collection permissions.